## Fixes applied

**Restored (this cell was entirely missing):** `clean_text`, `extract_skills` + `SKILLS_DB`, `identify_domain` + `DOMAIN_SKILLS`, `semantic_similarity`, `extract_resume_experience`/`extract_jd_experience` + scoring functions, the date-range parsing helpers (`_extract_named_section`, `merge_and_sum_years`, etc.) that the existing date-range cell depends on, `extract_education`/`education_level`, `extract_candidate_name` -- 19 functions total, referenced throughout the notebook but defined nowhere.

**Fixed in the aggregator:**
1. Missing `return {` -- the function fell from an `if` block straight into dict key-value pairs with no opening brace. Fatal syntax error; the notebook couldn't load at all.
2. `valid_insights` (built across ~15 lines: project-extraction regex, certifications, unmapped-skill fallbacks) was computed and then discarded -- the return dict used `llm_data.get("value_addition_insights", [])` directly instead.

**Restored:** `evaluate_candidate_report()` -- called in the final evaluation cell but never defined anywhere. Connects the extracted features to the trained Random Forest + SHAP explainer and produces every field `print_report()` expects.

**Two more bugs found only by running real uploaded files (Udit Yadav's resume against the Celebal JD), not visible from reading the code:**

6. **Section-boundary detection matched a header word inside ordinary prose.** The word "experience" appeared in Udit's Professional Summary sentence ("...skills and experience building responsive web interfaces...") -- the section-finder grabbed that occurrence instead of the real `EXPERIENCE` heading later in the resume, capturing a useless one-sentence fragment. Fixed: a section header must now be its own standalone line, not just a substring anywhere in the text.

7. **A "don't count Education dates" safety filter was too broad and started excluding legitimate experience.** It skipped any date range with "university" (among other words) nearby -- which correctly blocks a B.Tech degree's dates, but also wrongly blocked a real Teaching Assistant role because the *employer's name* was "JK Lakshmipat University." Removed: fix #6 already isolates the Experience section properly, so this filter was redundant protection that had become a source of false negatives instead.

In [1]:
# Cell 1: Environment Setup & Package Imports
import os
import sys
import json
import re
import time
import concurrent.futures
import datetime as _dt
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
import shap

# Try importing Ollama if installed locally
try:
    import ollama
    print("[+] Local Ollama package detected.")
except ImportError:
    print("[!] Note: 'ollama' Python package not found. Model will run in high-speed NLP + Rules + Random Forest mode.")

print("Environment initialized successfully.")


[+] Local Ollama package detected.
Environment initialized successfully.


In [2]:
# Cell 2: GLOBAL CONFIGURATION & HYPERPARAMETERS
RANDOM_STATE = 42
CONFIDENCE_THRESHOLD = 0.55
EXPERIENCE_UNKNOWN_SENTINEL = -1.0
OVERQUALIFICATION_RATIO_THRESHOLD = 1.75
MIN_SKILL_EVIDENCE = 3
MIN_RESUME_SKILLS = 2
HIGH_EXPERIENCE_REQUIREMENT_THRESHOLD = 5
OLLAMA_MODEL_NAME = "llama3.2"
OLLAMA_TIMEOUT_SECONDS = 45

FEATURES = [
    "skill_match_score",
    "matched_skill_count",
    "missing_skill_count",
    "experience_match_score",
    "experience_ratio",
    "resume_experience_mentioned",
    "jd_experience_mentioned",
    "unstated_experience_for_senior_role",
    "semantic_similarity_score",
    "domain_match",
]


In [ ]:
# ============================================================
# CORE NLP PIPELINE -- restored (this cell was missing entirely from the
# uploaded notebook; every function below is called throughout cells 3-8
# but none of them existed anywhere in the file).
# ============================================================

# ------------------------------------------------------------
# Text cleaning
# ------------------------------------------------------------
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = text.replace("&", " and ").replace("-", " ")
    text = re.sub(r'\S+@\S+|http\S+|www\S+|linkedin\.com/\S+|\+?\d[\d\s\-\(\)]{8,}\d|[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


# ------------------------------------------------------------
# Skills dictionary + matching
# ------------------------------------------------------------
SKILLS_DB = [
    "python", "java", "c", "c++", "c#", "javascript", "typescript", "go", "rust", "kotlin", "swift",
    "html", "css", "react", "reactjs", "angular", "vue", "nodejs", "node.js", "express", "django",
    "flask", "fastapi", "spring", "spring boot", "tailwindcss", "bootstrap", "next.js", "nextjs",
    "sql", "mysql", "postgresql", "mongodb", "redis", "oracle", "sqlite", "firebase",
    "aws", "azure", "gcp", "docker", "kubernetes", "terraform", "jenkins", "ci/cd", "git", "github",
    "machine learning", "deep learning", "artificial intelligence", "nlp", "computer vision",
    "tensorflow", "pytorch", "keras", "scikit-learn", "pandas", "numpy", "opencv",
    "llms", "rag", "agentic ai", "ai agents", "langchain", "vector databases",
    "rest apis", "graphql", "microservices", "figma", "ui/ux", "adobe xd",
    "data analysis", "data visualization", "power bi", "tableau", "excel",
    "hadoop", "spark", "kafka", "airflow", "etl",
    "linux", "bash", "shell scripting", "networking", "cybersecurity",
    "problem solving", "team leadership", "communication", "project management", "agile", "scrum",

    # Finance / Accounting / Audit -- added because the original list was 100% tech-hiring
    # vocabulary, which made skill_match_score and identify_domain meaningless for any
    # non-tech resume/JD pair (confirmed: a Chartered Accountant resume against a CA audit
    # JD matched on "communication" alone -- 0% otherwise, despite obvious relevance).
    "chartered accountant", "ca", "ca final", "ca intermediate", "articleship",
    "statutory audit", "internal audit", "tax audit", "auditing", "assurance",
    "accounting standards", "ind as", "ifrs", "gaap", "schedule iii",
    "income tax", "gst", "tds", "tcs", "roc filing", "companies act",
    "financial modelling", "financial analysis", "valuation", "dcf", "wacc", "irr",
    "cma report", "credit appraisal", "due diligence", "risk assessment", "compliance",
    "sap", "tally", "tally prime", "compuoffice", "quickbooks", "erp",
    "ms office", "microsoft excel", "microsoft powerpoint", "advanced excel",
]


def extract_skills(clean_resume_text: str) -> list:
    found = []
    for skill in SKILLS_DB:
        skill_clean = clean_text(skill)
        pattern = r"\b" + re.escape(skill_clean).replace(r"\ ", r"\s+") + r"\b"
        if re.search(pattern, clean_resume_text):
            found.append(skill)
    # fuzzy pass for near-misses (typos, slight variants) using rapidfuzz if available
    try:
        from rapidfuzz import fuzz
        tokens = set(clean_resume_text.split())
        for skill in SKILLS_DB:
            if skill in found:
                continue
            skill_tokens = clean_text(skill).split()
            if len(skill_tokens) == 1:
                for tok in tokens:
                    if len(tok) > 3 and fuzz.ratio(tok, skill_tokens[0]) >= 90:
                        found.append(skill)
                        break
    except ImportError:
        pass
    return sorted(set(found))


def get_matched_skills(resume_skills: list, jd_skills: list) -> list:
    return sorted(set(s.lower() for s in resume_skills) & set(s.lower() for s in jd_skills))


def get_missing_skills(resume_skills: list, jd_skills: list) -> list:
    return sorted(set(s.lower() for s in jd_skills) - set(s.lower() for s in resume_skills))


def skill_match_score(resume_skills: list, jd_skills: list) -> float:
    if not jd_skills:
        return 0.0
    matched = get_matched_skills(resume_skills, jd_skills)
    return round(len(matched) / len(jd_skills), 4)


# ------------------------------------------------------------
# Domain identification
# ------------------------------------------------------------
DOMAIN_SKILLS = {
    "Software Development": {"python", "java", "c", "c++", "c#", "javascript", "typescript", "react",
                              "reactjs", "angular", "vue", "nodejs", "node.js", "express", "django",
                              "flask", "fastapi", "spring", "spring boot", "html", "css", "rest apis",
                              "graphql", "microservices", "git", "github"},
    "Data Science / AI": {"machine learning", "deep learning", "artificial intelligence", "nlp",
                           "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn",
                           "pandas", "numpy", "opencv", "llms", "rag", "agentic ai", "ai agents",
                           "langchain", "vector databases", "data analysis", "data visualization"},
    "Cloud / DevOps": {"aws", "azure", "gcp", "docker", "kubernetes", "terraform", "jenkins",
                        "ci/cd", "linux", "bash", "shell scripting"},
    "Database": {"sql", "mysql", "postgresql", "mongodb", "redis", "oracle", "sqlite", "firebase"},
    "Design": {"figma", "ui/ux", "adobe xd", "tailwindcss", "bootstrap"},
    "Finance / Accounting / Audit": {
        "chartered accountant", "ca", "ca final", "ca intermediate", "articleship",
        "statutory audit", "internal audit", "tax audit", "auditing", "assurance",
        "accounting standards", "ind as", "ifrs", "gaap", "schedule iii",
        "income tax", "gst", "tds", "tcs", "roc filing", "companies act",
        "financial modelling", "financial analysis", "valuation", "dcf", "wacc", "irr",
        "cma report", "credit appraisal", "due diligence", "risk assessment", "compliance",
        "sap", "tally", "tally prime", "quickbooks",
    },
}


def identify_domain(skills: list) -> str:
    skills_lower = set(s.lower() for s in skills)
    best_domain, best_overlap = "Unknown", 0
    for domain, domain_set in DOMAIN_SKILLS.items():
        overlap = len(skills_lower & domain_set)
        if overlap > best_overlap:
            best_domain, best_overlap = domain, overlap
    return best_domain


# ------------------------------------------------------------
# Semantic similarity (SBERT, falling back to TF-IDF if unavailable)
# ------------------------------------------------------------
_SBERT_MODEL = None
_SBERT_LOAD_ATTEMPTED = False


def _get_sbert_model():
    global _SBERT_MODEL, _SBERT_LOAD_ATTEMPTED
    if _SBERT_LOAD_ATTEMPTED:
        return _SBERT_MODEL
    _SBERT_LOAD_ATTEMPTED = True
    try:
        from sentence_transformers import SentenceTransformer
        _SBERT_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
    except Exception as e:
        print(f"[WARNING] Could not load SBERT model ({e}). Falling back to TF-IDF cosine similarity.")
        _SBERT_MODEL = None
    return _SBERT_MODEL


def semantic_similarity(resume_clean: str, jd_clean: str) -> float:
    model = _get_sbert_model()
    if model is not None:
        try:
            from sentence_transformers import util
            emb = model.encode([resume_clean, jd_clean], convert_to_tensor=True)
            return round(float(util.cos_sim(emb[0], emb[1])), 4)
        except Exception:
            pass
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity
        if not resume_clean.strip() or not jd_clean.strip():
            return 0.0
        vec = TfidfVectorizer().fit_transform([resume_clean, jd_clean])
        return round(float(cosine_similarity(vec[0], vec[1])[0][0]), 4)
    except Exception:
        return 0.0


# ------------------------------------------------------------
# Experience: explicit-phrase regex (Tier 2 fallback), JD parsing, scoring
# ------------------------------------------------------------
_RESUME_EXP_PATTERNS = [re.compile(r'(\d+)\+?\s*years'), re.compile(r'(\d+)\+?\s*year'),
                        re.compile(r'(\d+)\s*yrs'), re.compile(r'(\d+)\s*yr')]
_JD_EXP_PATTERNS = [re.compile(r'experience\s*:?\s*(\d+)\+?\s*years'),
                    re.compile(r'experience\s*:?\s*(\d+)\+?\s*year'),
                    re.compile(r'(\d+)\+?\s*years'), re.compile(r'(\d+)\+?\s*year')]


def extract_resume_experience(text: str):
    text = str(text).lower()
    for pattern in _RESUME_EXP_PATTERNS:
        match = pattern.search(text)
        if match:
            return int(match.group(1))
    return np.nan


def extract_jd_experience(text: str):
    text = str(text).lower()
    for pattern in _JD_EXP_PATTERNS:
        match = pattern.search(text)
        if match:
            return int(match.group(1))
    return np.nan


def calculate_experience_match_score(resume_exp, jd_exp):
    if pd.isna(resume_exp) or pd.isna(jd_exp) or jd_exp == 0:
        return np.nan
    return round(min(resume_exp / jd_exp, 1.0), 4)


def calculate_experience_ratio(resume_exp, jd_exp):
    if pd.isna(resume_exp) or pd.isna(jd_exp) or jd_exp == 0:
        return np.nan
    return round(resume_exp / jd_exp, 4)


def is_overqualified(experience_ratio) -> bool:
    if pd.isna(experience_ratio):
        return False
    return experience_ratio >= OVERQUALIFICATION_RATIO_THRESHOLD


def experience_unknown_for_senior_role(resume_exp_mentioned: int, jd_exp) -> int:
    if resume_exp_mentioned:
        return 0
    if pd.isna(jd_exp):
        return 0
    return int(jd_exp >= HIGH_EXPERIENCE_REQUIREMENT_THRESHOLD)


def impute_experience_score(scores: pd.Series, fill_value: float = EXPERIENCE_UNKNOWN_SENTINEL) -> pd.Series:
    return scores.fillna(fill_value)


# ------------------------------------------------------------
# Date-range experience parsing helpers
# (used by extract_experience_from_date_ranges, already defined elsewhere
# in this notebook -- that function calls these by name)
# ------------------------------------------------------------
_MULTILINGUAL_MONTHS_MAP = {
    "jan": 1, "january": 1, "feb": 2, "february": 2, "mar": 3, "march": 3,
    "apr": 4, "april": 4, "may": 5, "jun": 6, "june": 6, "jul": 7, "july": 7,
    "aug": 8, "august": 8, "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10, "nov": 11, "november": 11, "dec": 12, "december": 12,
    # NOTE: named "multilingual" for future extension -- currently English
    # only. Add transliterated/native month names here as needed, e.g.
    # Hindi "जनवरी": 1, romanized "janvari": 1, etc.
}
_CURRENT_WORDS = {"present", "till now", "till date", "current", "ongoing",
                   "now", "date", "continuing", "till present"}

_EXPERIENCE_SECTION_HEADERS = [
    r"employment\s*history", r"work\s*experience", r"professional\s*experience",
    r"experience", r"career\s*history", r"work\s*history",
]
_NON_EXPERIENCE_SECTION_HEADERS = [
    r"education", r"achievements?", r"awards?", r"publications?",
    r"skills?", r"courses?\s*taught", r"projects?", r"references?",
    r"certifications?", r"workshops?", r"conferences?", r"summer\s*school",
    r"fdp", r"training", r"objective", r"summary", r"profile",
]

_MONTH_NAME_ALT = "|".join(sorted(_MULTILINGUAL_MONTHS_MAP.keys(), key=len, reverse=True))
# NOTE: re.escape() is load-bearing here, not cosmetic -- this pattern uses
# re.VERBOSE, which silently strips *unescaped* whitespace. Without
# escaping, "till now" becomes the literal "tillnow" in the compiled
# pattern and will never match real text.
_CURRENT_ALT = "|".join(re.escape(w) for w in sorted(_CURRENT_WORDS, key=len, reverse=True))

_MONTHNAME_RANGE_RE = re.compile(rf"""
    (?:(?P<m1>{_MONTH_NAME_ALT})\s+)?(?P<y1>(19|20)\d{{2}})
    (?:\s*(?:to|[-–—]+)\s*|\s{{2,}})
    (?:(?:(?P<m2>{_MONTH_NAME_ALT})\s+)?(?P<y2>(19|20)\d{{2}}) | (?P<current>{_CURRENT_ALT}))
    """, re.IGNORECASE | re.VERBOSE)


def _extract_named_section(text: str, start_headers, end_headers) -> str:
    # FIX: this originally used re.search() against the whole text, which
    # matches a header word ANYWHERE -- including inside ordinary prose.
    # Confirmed on a real resume: "...skills and experience building
    # responsive web interfaces" in a Professional Summary sentence
    # matched the bare "experience" pattern before the real EXPERIENCE
    # section header later in the document, so the function grabbed a
    # useless one-sentence fragment instead of the actual section.
    # Fixed: a header must be its own standalone line (optionally with a
    # trailing colon), not merely a substring of a sentence.
    lines = text.split("\n")
    start_pat = re.compile(r"^\s*(?:" + "|".join(start_headers) + r")\s*:?\s*$", re.IGNORECASE)
    end_pat = re.compile(r"^\s*(?:" + "|".join(end_headers) + r")\s*:?\s*$", re.IGNORECASE)

    start_idx = next((i for i, line in enumerate(lines) if start_pat.match(line)), None)
    if start_idx is None:
        return ""
    end_idx = next((i for i in range(start_idx + 1, len(lines)) if end_pat.match(lines[i])), len(lines))
    return "\n".join(lines[start_idx + 1:end_idx])


def merge_and_sum_years(intervals: list):
    """Merges overlapping/contiguous (start, end) date ranges so concurrent
    roles aren't double-counted, and returns total years -- or None if
    empty. Shared by both the NLP date-range parser and the LLM-fallback
    path, so there is exactly ONE place that does this arithmetic."""
    if not intervals:
        return None
    intervals = sorted(intervals, key=lambda iv: iv[0])
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    total_days = sum((end - start).days for start, end in merged)
    return round(total_days / 365.25, 2)


# ------------------------------------------------------------
# Education
# (patterns match clean_text() output, which strips periods -- "B.Tech"
# arrives as "b tech", not "b.tech")
# ------------------------------------------------------------
_EDUCATION_PATTERNS = [
    (r"\bphd\b|\bph\s+d\b|\bdoctorate\b", "PhD"),
    (r"\bchartered accountant\b|\bca final\b|\b(?<!\w)ca(?!\w)\b.{0,15}\bicai\b", "CA (Chartered Accountant)"),
    (r"\bm\s*tech\b|\bmaster of technology\b|\bmaster of engineering\b", "M.Tech"),
    (r"\bmba\b", "MBA"),
    (r"\bmca\b", "MCA"),
    (r"\bm\s*com\b|\bmaster of commerce\b", "M.Com"),
    (r"\bm\s*sc\b|\bmaster of science\b", "M.Sc"),
    (r"\bmaster of arts\b", "M.A"),
    (r"\bb\s*tech\b|\bbachelor of technology\b|\bbachelor of engineering\b", "B.Tech"),
    (r"\bbca\b", "BCA"),
    (r"\bb\s*com\b|\bbachelor of commerce\b", "B.Com"),
    (r"\bb\s*sc\b|\bbachelor of science\b", "B.Sc"),
    (r"\bbachelor of arts\b", "B.A"),
    (r"\bdiploma\b", "Diploma"),
    (r"\bhigh school\b|\b12th\b|\bhsc\b", "High School"),
]
_EDU_PATTERNS_COMPILED = [(re.compile(p, re.IGNORECASE), label) for p, label in _EDUCATION_PATTERNS]
_EDUCATION_RANK = {"High School": 1, "Diploma": 2, "BCA": 3, "B.A": 3, "B.Sc": 3, "B.Tech": 3, "B.Com": 3,
                    "MCA": 4, "M.A": 4, "M.Sc": 4, "M.Tech": 4, "MBA": 4, "M.Com": 4,
                    "CA (Chartered Accountant)": 4, "PhD": 5}


def extract_education(text: str) -> list:
    found = []
    for pattern, label in _EDU_PATTERNS_COMPILED:
        if pattern.search(text) and label not in found:
            found.append(label)
    return found


def education_level(education_list: list) -> str:
    if not education_list:
        return "Not specified"
    return max(education_list, key=lambda e: _EDUCATION_RANK.get(e, 0))


# ------------------------------------------------------------
# Candidate name (heuristic: first plausible name-shaped line)
# ------------------------------------------------------------
def extract_candidate_name(resume_text_raw: str) -> str:
    lines = [l.strip() for l in str(resume_text_raw).splitlines() if l.strip()]
    skip_words = {"resume", "cv", "curriculum vitae", "summary", "objective", "profile"}
    # Common signature/qualification suffixes that trail a name at the bottom
    # of a resume, e.g. "SUJAL AGRAWAL\nChartered Accountant ACA, BCom." --
    # stripped so the returned name doesn't absorb them.
    _qualification_words = {"chartered", "accountant", "aca", "bcom", "b.com",
                             "mba", "ca", "cpa", "cfa", "engineer", "developer"}

    def _looks_like_name(line: str) -> bool:
        if "@" in line or re.search(r"\d{3,}", line):
            return False
        if line.lower() in skip_words:
            return False
        words = line.split()
        if not (1 < len(words) <= 5 and all(w[0].isupper() or not w[0].isalpha() for w in words if w)):
            return False
        # A line can be "Title Case" without being a person's name, e.g. a
        # trailing qualification/signature line like "Chartered Accountant
        # ACA, BCom." Reject lines whose words are mostly qualification
        # terms rather than a plausible personal name.
        stripped = [w.strip(",.").lower() for w in words]
        if any(w in _qualification_words for w in stripped):
            return False
        return True

    # FIX: name is usually near the top, but not always -- some resumes
    # (as seen with a real upload) only place it as a signature line near
    # the bottom. Check the first 6 lines first, then fall back to the
    # last 6 before giving up, instead of assuming top-of-document only.
    for line in lines[:6]:
        if _looks_like_name(line):
            return line.title()
    for line in reversed(lines[-6:]):
        if _looks_like_name(line):
            return line.title()
        # Handle "NAME, Qualification" or "NAME\nQualification" signature
        # lines by stripping a trailing qualification clause and re-checking.
        head = re.split(r"[,\n]", line)[0].strip()
        if head != line and _looks_like_name(head):
            return head.title()
    return "Candidate"


# ------------------------------------------------------------
# FEATURE_LABELS -- human-readable names for SHAP explanations
# ------------------------------------------------------------
FEATURE_LABELS = {
    "skill_match_score": "Skill Match",
    "matched_skill_count": "Number of Matched Skills",
    "missing_skill_count": "Number of Missing Skills",
    "experience_match_score": "Experience Match",
    "experience_ratio": "Experience Ratio (uncapped)",
    "resume_experience_mentioned": "Resume States Experience",
    "jd_experience_mentioned": "JD States Experience Requirement",
    "unstated_experience_for_senior_role": "Unstated Experience Risk (Senior Role)",
    "semantic_similarity_score": "Semantic Similarity",
    "domain_match": "Domain Match",
}


In [4]:
# Cell 3: LAYOUT-AWARE MULTI-FORMAT DOCUMENT EXTRACTOR
class ScannedPDFError(Exception): pass
class CorruptPDFError(Exception): pass

_GLUED_WORD_RE = re.compile(r"[a-zA-Z]{16,}")
_CID_ARTIFACT_RE = re.compile(r"\(cid:\d+\)")

def _extract_page_text_adaptive(page) -> str:
    candidates = []
    for tol in (1, 2, 3):
        try:
            text = page.extract_text(x_tolerance=tol) or ""
        except Exception:
            continue
        candidates.append((tol, text, len(_GLUED_WORD_RE.findall(text))))
    if not candidates:
        return ""
    candidates.sort(key=lambda c: (c[2], -c[0]))
    return candidates[0][1]

def extract_text_from_pdf(pdf_path: str) -> str:
    try:
        import fitz  # PyMuPDF
        doc = fitz.open(pdf_path)
        text_parts = [page.get_text("text", sort=True) for page in doc]
        full_text = "\n".join(t for t in text_parts if t.strip())
        if full_text.strip():
            return _CID_ARTIFACT_RE.sub(" ", full_text)
    except Exception:
        pass
        
    import pdfplumber
    try:
        with pdfplumber.open(pdf_path) as pdf:
            text_parts = [_extract_page_text_adaptive(page) for page in pdf.pages]
    except Exception as e:
        raise CorruptPDFError(f"Could not open '{pdf_path}': {e}") from e

    full_text = "\n".join(t for t in text_parts if t)
    full_text = _CID_ARTIFACT_RE.sub(" ", full_text)
    if full_text.strip():
        return full_text

    try:
        from pdf2image import convert_from_path
        import pytesseract
        pages = convert_from_path(pdf_path)
        ocr_parts = [pytesseract.image_to_string(p) for p in pages]
        return "\n".join(t for t in ocr_parts if t)
    except Exception:
        raise ScannedPDFError(f"No text layer or OCR available for '{pdf_path}'.")

def extract_text_from_docx(docx_path: str) -> str:
    import docx
    doc = docx.Document(docx_path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip(): parts.append(cell.text)
    return "\n".join(parts)

def extract_text_from_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf": return extract_text_from_pdf(file_path)
    elif ext == ".docx": return extract_text_from_docx(file_path)
    else:
        from PIL import Image
        import pytesseract
        return pytesseract.image_to_string(Image.open(file_path))

def safe_extract_text_from_file(file_path: str):
    try:
        return extract_text_from_file(file_path), None
    except Exception as e:
        return None, str(e)


In [5]:
def extract_experience_from_date_ranges(text: str):
    text_str = str(text)
    section = _extract_named_section(text_str, _EXPERIENCE_SECTION_HEADERS, _NON_EXPERIENCE_SECTION_HEADERS)
    if not section:
        return 0.0
    intervals, claimed = [], []
    today = _dt.date.today()
    for m in _MONTHNAME_RANGE_RE.finditer(section):
        span = m.span()
        if any(s0 < span[1] and span[0] < s1 for s0, s1 in claimed): continue
        # FIX: this used to also check for education-related keywords
        # ("university", "b.tech", "cgpa", etc.) within 80 characters of
        # each date match and skip it if found -- intended to stop
        # Education-section dates leaking into the experience total. But
        # confirmed on a real resume this was too broad: a Teaching
        # Assistant role's date range got silently excluded because its
        # EMPLOYER's name ("JK Lakshmipat University") happened to contain
        # "university" -- the same word the filter was watching for.
        # Removed: _extract_named_section() now correctly isolates the
        # Experience section by requiring standalone header lines, so
        # Education dates can no longer leak in through this path at all
        # -- this filter was redundant protection that only produced false
        # negatives once that fix was in place.
        try:
            m1_str = m.group("m1")
            mo1 = _MULTILINGUAL_MONTHS_MAP.get(m1_str.lower(), 1) if (m1_str and isinstance(m1_str, str)) else 1
            start = _dt.date(int(m.group("y1")), mo1, 1)
            if m.group("current"):
                end = today
            else:
                m2_str = m.group("m2")
                mo2 = _MULTILINGUAL_MONTHS_MAP.get(m2_str.lower(), 12) if (m2_str and isinstance(m2_str, str)) else 12
                end = _dt.date(int(m.group("y2")), mo2, 1)
            if end.year > today.year or (end.year == today.year and end.month > today.month):
                continue
            if end >= start:
                claimed.append(span)
                intervals.append((start, end))
        except Exception:
            continue
    res = merge_and_sum_years(intervals)
    return res if res is not None else 0.0


In [ ]:
# Cell 5: PARALLEL TRACK 2: OLLAMA LLM ENGINE WITH NATIVE JSON GUARANTEE
#
# FIXES APPLIED HERE (found by tracing a real CA-resume-vs-CA-JD run):
#
# 1. The JD text was accepted as a parameter (`jd_text`) but never actually
#    sent to the model -- yet the schema asked for JD-relative fields
#    ("value NOT required by the JD"). The model was answering blind.
#    Fixed: the JD is now included in the user message, and the prompt is
#    explicit that both documents are provided.
# 2. `unmapped_skills` asked the model to find things "NOT in a standard
#    basic skills list" without ever showing it that list, and the four
#    examples given (vLLM, DeepSeek, FlashAttention, BioPython) were all
#    ML/software tooling -- steering every answer toward tech regardless of
#    the resume's actual field. Fixed: the real SKILLS_DB is now included,
#    and the examples are domain-neutral.
# 3. `dynamic_recommendations` was requested from the model but never read
#    anywhere downstream (the aggregator builds its own recommendations
#    from the rule-based missing_skills list instead) -- dropped from the
#    schema so the model isn't wasting output generating something unused.
# 4. Date format compliance ("YYYY-MM or YYYY") was requested but never
#    enforced -- compute_experience_years_from_entries() below now accepts
#    several common variants instead of silently discarding anything else.
# 5. resume_text was hard-truncated to the first 4000 characters, which
#    risks losing content near the end of longer resumes (confirmed: a real
#    resume's signature name and trailing certifications sat right at that
#    boundary). Fixed: truncation now keeps the head AND the tail instead of
#    just the head, and the limit only kicks in for resumes that actually
#    exceed it.

EXACT_LLM_PROMPT = """You are an expert HR AI evaluator. You will be given a
JOB DESCRIPTION and a RESUME. Extract information from the resume that
traditional regex/NLP might miss, using the job description as context for
what is or isn't already required. Return ONLY a raw JSON object matching
exactly this schema:

{
  "candidate_name": "extracted full name or empty string",
  "experience_entries": [
    {"role": "string", "organization": "string", "start_date": "YYYY-MM or YYYY", "end_date": "YYYY-MM, YYYY, or the literal string present if current"}
  ],
  "certifications": [
    {"name": "string", "issuer": "string or null"}
  ],
  "unmapped_skills": ["skills/tools/qualifications the candidate demonstrably has that are NOT already covered by the KNOWN_SKILLS_LIST provided below -- these can be from any field (e.g. technical, financial, legal, medical, creative), not just software"],
  "value_addition_insights": ["3-4 bullet points on candidate value NOT required by the job description above -- e.g. research papers, patents, fellowships, hackathons, leadership, published work, competitions, workshops attended"]
}

Rules:
- experience_entries means paid/professional roles, articleships, and research
  fellowships -- NOT study years themselves, NOT awards, NOT workshops.
- ALWAYS format start_date and end_date as "YYYY-MM" if a month is stated,
  or "YYYY" if only a year is stated. Never return a month name as text
  (e.g. write "2023-09", not "September 2023").
- Output nothing but valid JSON."""


def _build_llm_resume_snippet(resume_text: str, head_chars: int = 4000, tail_chars: int = 1500) -> str:
    """Keep the head AND the tail of a long resume instead of just the head,
    so a name/signature or certifications list near the end doesn't get
    silently dropped for resumes longer than the head cap."""
    text = str(resume_text)
    if len(text) <= head_chars + tail_chars:
        return text
    return text[:head_chars] + "\n...[middle of resume omitted for length]...\n" + text[-tail_chars:]


def run_ollama_parallel_track(resume_text: str, jd_text: str) -> dict:
    try:
        import ollama
        resume_snippet = _build_llm_resume_snippet(resume_text)
        known_skills = ", ".join(SKILLS_DB)
        user_content = (
            f"JOB DESCRIPTION:\n{jd_text[:3000]}\n\n"
            f"KNOWN_SKILLS_LIST (already detected separately, do not repeat these in unmapped_skills):\n{known_skills}\n\n"
            f"RESUME:\n{resume_snippet}"
        )
        res = ollama.chat(
            model=OLLAMA_MODEL_NAME,
            messages=[
                {"role": "system", "content": EXACT_LLM_PROMPT},
                {"role": "user", "content": user_content}
            ],
            format="json"
        )
        content = res["message"]["content"]
        data = json.loads(content)
        return {
            "candidate_name": str(data.get("candidate_name", "")).strip(),
            "experience_entries": data.get("experience_entries", []) or [],
            "certifications": data.get("certifications", []) or [],
            "unmapped_skills": [str(s).lower().strip() for s in data.get("unmapped_skills", []) if s],
            "value_addition_insights": [str(v).strip() for v in data.get("value_addition_insights", []) if v],
            "llm_status": f"Ollama {OLLAMA_MODEL_NAME} (JSON mode)",
        }
    except Exception as e:
        return {
            "candidate_name": "",
            "experience_entries": [],
            "certifications": [],
            "unmapped_skills": [],
            "value_addition_insights": [f"Ollama LLM track offline/unavailable ({e}) — running in high-speed NLP fallback mode."],
            "llm_status": f"UNAVAILABLE ({e})",
        }


def compute_experience_years_from_entries(entries: list):
    # FIX: only accepted strict "YYYY-MM" / "YYYY" strings before, silently
    # dropping any entry where the model didn't comply exactly with that
    # format (e.g. "September 2023", "Sep 2023", "09/2023"). Now tries
    # several common variants before giving up on an entry.
    def _parse(d_str):
        if not d_str or str(d_str).strip().lower() in _CURRENT_WORDS:
            return _dt.date.today()
        s = str(d_str).strip()
        m = re.match(r"^(\d{4})-(\d{1,2})$", s)
        if m: return _dt.date(int(m.group(1)), int(m.group(2)), 1)
        m = re.match(r"^(\d{4})/(\d{1,2})$", s)
        if m: return _dt.date(int(m.group(1)), int(m.group(2)), 1)
        m = re.match(r"^(\d{1,2})[/-](\d{4})$", s)
        if m: return _dt.date(int(m.group(2)), int(m.group(1)), 1)
        m = re.match(r"^(\d{4})$", s)
        if m: return _dt.date(int(m.group(1)), 1, 1)
        m = re.match(rf"^({_MONTH_NAME_ALT})[\s\-,]+(\d{{4}})$", s, re.IGNORECASE)
        if m: return _dt.date(int(m.group(2)), _MULTILINGUAL_MONTHS_MAP.get(m.group(1).lower(), 1), 1)
        m = re.match(rf"^(\d{{4}})[\s\-,]+({_MONTH_NAME_ALT})$", s, re.IGNORECASE)
        if m: return _dt.date(int(m.group(1)), _MULTILINGUAL_MONTHS_MAP.get(m.group(2).lower(), 1), 1)
        return None
    intervals = []
    for entry in entries:
        st = _parse(entry.get("start_date"))
        en = _dt.date.today() if str(entry.get("end_date", "")).lower() in _CURRENT_WORDS else _parse(entry.get("end_date"))
        if st is None or en is None or en < st: continue
        intervals.append((st, en))
    return merge_and_sum_years(intervals)


In [ ]:
# Cell 6: PARALLEL HYBRID AGGREGATOR
def run_parallel_hybrid_pipeline(resume_text_raw: str, jd_text_raw: str) -> dict:
    resume_clean = clean_text(resume_text_raw)
    jd_clean = clean_text(jd_text_raw)
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        future_nlp = executor.submit(extract_skills, resume_clean)
        future_llm = executor.submit(run_ollama_parallel_track, resume_text_raw, jd_text_raw)
        
        nlp_skills = future_nlp.result()
        try:
            llm_data = future_llm.result(timeout=OLLAMA_TIMEOUT_SECONDS)
        except concurrent.futures.TimeoutError:
            llm_data = {
                "candidate_name": "", "experience_entries": [], "certifications": [],
                "unmapped_skills": [], "value_addition_insights": [f"Ollama timed out after {OLLAMA_TIMEOUT_SECONDS}s."],
                "dynamic_recommendations": [], "llm_status": "TIMEOUT"
            }
            
    unmapped = set(llm_data.get("unmapped_skills", []))
    resume_skills = sorted(list(set(nlp_skills) | unmapped))
    
    jd_skills = extract_skills(jd_clean)
    matched_skills = get_matched_skills(resume_skills, jd_skills)
    missing_skills = get_missing_skills(resume_skills, jd_skills)
    skill_score = skill_match_score(resume_skills, jd_skills)
    
    date_exp = extract_experience_from_date_ranges(resume_text_raw)
    regex_exp = extract_resume_experience(resume_clean)
    
    if date_exp is not None and date_exp > 0:
        resume_exp = date_exp
        exp_source = "NLP Date-Range Parser (Tier 1)"
    elif not pd.isna(regex_exp) and regex_exp > 0:
        resume_exp = regex_exp
        exp_source = "NLP Explicit Phrase (Tier 2)"
    else:
        llm_exp = compute_experience_years_from_entries(llm_data.get("experience_entries", []))
        if llm_exp is not None and llm_exp > 0:
            resume_exp = llm_exp
            exp_source = "LLM-extracted dates, NLP-computed total (Tier 3)"
        else:
            resume_exp = np.nan
            exp_source = "Unverified (Sentinel -1.0)"
            
    jd_exp = extract_jd_experience(jd_clean)
    exp_score = calculate_experience_match_score(resume_exp, jd_exp)
    exp_ratio = calculate_experience_ratio(resume_exp, jd_exp)
    overqualified = is_overqualified(exp_ratio)
    resume_exp_mentioned = int(not pd.isna(resume_exp))
    jd_exp_mentioned = int(not pd.isna(jd_exp))
    unstated_exp = experience_unknown_for_senior_role(resume_exp_mentioned, jd_exp)
    
    resume_edu = extract_education(resume_clean)
    resume_edu_level = education_level(resume_edu)
    resume_domain = identify_domain(resume_skills)
    jd_domain = identify_domain(jd_skills)
    domain_match_val = int(resume_domain == jd_domain and resume_domain != "Unknown")
    sim_score = semantic_similarity(resume_clean, jd_clean)
    
    candidate_name = llm_data.get("candidate_name") or extract_candidate_name(resume_text_raw)
    LEARNING_RESOURCE_MAP = {
        "agentic ai": "Agentic AI & AI Agents Masterclass (DeepLearning.AI / LangChain)",
        "ai agents": "Building Autonomous AI Agents (DeepLearning.AI / AutoGen)",
        "rag": "Retrieval Augmented Generation with LangChain & LlamaIndex (Coursera)",
        "llms": "Generative AI & LLMs Architecture (Hugging Face / DeepLearning.AI)",
        "figma": "Figma UI/UX Design Essentials (Udemy)",
        "rest apis": "RESTful API Design & Building with Node.js & Express (freeCodeCamp)",
        "nodejs": "Node.js & Express Complete Developer Course (Udemy)",
        "mongodb": "MongoDB Developer Certification & Data Modeling",
        "react": "React - The Complete Guide (Udemy)",
        "docker": "Docker Certified Associate (DCA)",
        "kubernetes": "Certified Kubernetes Administrator (CKA)",
        "aws": "AWS Certified Solutions Architect (AWS Skill Builder)",
        "sql": "SQL for Data Science & Analytics (Coursera)",
        "git": "Git & GitHub for Beginners (freeCodeCamp)",
        "python": "Python for Everybody Specialization (Coursera)",
        "machine learning": "Machine Learning Specialization (Andrew Ng, Coursera)",
        "quickbooks": "QuickBooks Online Certified User (Intuit)",
        "sap": "SAP ERP Financial Accounting (SAP Learning Hub)",
        "cfa": "CFA Institute Level 1 Certification",
        # Finance / Accounting / Audit -- added alongside the SKILLS_DB
        # expansion so recommendations for this domain aren't just the
        # generic "Certification & Project in X" fallback string.
        "auditing": "Auditing Standards & Practice (ICAI CPE Course)",
        "statutory audit": "Statutory Audit Certificate Course (ICAI)",
        "internal audit": "Certified Internal Auditor (CIA), IIA",
        "communication": "Business Communication & Presentation Skills (Coursera)",
        "gst": "GST Certification Course (ICAI / ClearTax Academy)",
        "ind as": "Ind AS Certification Course (ICAI)",
        "ifrs": "IFRS Certificate (ACCA / ICAI)",
        "financial modelling": "Financial Modelling & Valuation Analyst (FMVA), CFI",
        "valuation": "Business Valuation Certification (CFI)",
    }
    recs = []
    if missing_skills:
        for s in missing_skills[:6]:
            res_course = LEARNING_RESOURCE_MAP.get(str(s).lower().strip(), f"Certification & Project in {s.title()}")
            recs.append({"skill": s, "resource": res_course})
    raw_insights = llm_data.get("value_addition_insights", [])
    valid_insights = [v for v in raw_insights if isinstance(v, str) and not v.startswith("Ollama LLM track") and not v.startswith("Ollama timed out") and len(v.strip()) > 3]

    # FIX: certifications previously came ONLY from the LLM track
    # (llm_data.get("certifications", [])), so with Ollama unavailable the
    # certifications field was silently empty even when the resume has an
    # explicit Certifications section. Add a regex-based fallback that
    # mirrors how education/experience already get a fallback.
    certs = llm_data.get("certifications", [])
    if not certs:
        cert_section = _extract_named_section(resume_text_raw, [r"certifications?"], _EXPERIENCE_SECTION_HEADERS + [h for h in _NON_EXPERIENCE_SECTION_HEADERS if h != r"certifications?"])
        if cert_section:
            _date_only_re = re.compile(r"^(?:(?:" + _MONTH_NAME_ALT + r")[\s,]*)?(?:19|20)\d{2}$", re.IGNORECASE)
            _name_line_re = re.compile(r"^[A-Za-z.,]+(?:\s+[A-Za-z.,]+){0,3}$")
            raw_lines = [l.strip() for l in cert_section.split("\n") if l.strip()]
            # FIX: with no section header after Certifications, this grabbed
            # everything to the end of the file, including a trailing
            # name/qualification signature block (e.g. "SUJAL AGRAWAL" /
            # "Chartered Accountant ACA, BCom."). Stop as soon as we see a
            # short, comma-free, all-title-case line with no digits and no
            # "By <issuer>" pattern -- that's a signature, not a cert.
            kept_lines = []
            for line in raw_lines:
                if _name_line_re.match(line) and "," not in line and " by " not in line.lower() and not re.search(r"\d", line):
                    break
                kept_lines.append(line)
            # FIX: certification text commonly line-wraps mid-phrase (e.g.
            # "...Jun 2026, Advanced" / "Microsoft Excel By Skill91, Jul
            # 2023" on the next physical line purely due to page width).
            # Processing line-by-line split those into two fake
            # certifications ("Advanced" and "Microsoft Excel By Skill91").
            # Joining into one continuous string before splitting on commas
            # undoes the wrap so phrases aren't broken mid-word.
            joined = " ".join(kept_lines)
            raw_items = re.split(r",\s*(?=[A-Z])", joined)
            # Naive comma-splitting also produces standalone date fragments
            # (e.g. "Jun 2026", "2026") as their own fake "certifications" --
            # merge any fragment that is JUST a date back onto the previous
            # real certification instead.
            for item in raw_items:
                item = item.strip(" -\u2022\t")
                if not item:
                    continue
                if _date_only_re.match(item) and certs:
                    certs[-1]["name"] = f"{certs[-1]['name']} ({item})"
                elif len(item) > 3:
                    certs.append({"name": item, "issuer": None})

    if not valid_insights:
        valid_insights = []
        # FIX: added \b word boundaries so "project" can no longer match
        # inside the plural section header "PROJECTS" itself -- that used to
        # capture the header's trailing "S" plus the next line as a garbled
        # "project name".
        p_matches = re.findall(r'\b(?:project|developed|engineered|built)\b\s*[:|-]?\s*([A-Z0-9][A-Za-z0-9\s\-_]{3,40})', resume_text_raw, re.IGNORECASE)
        if p_matches:
            clean_projects = list(dict.fromkeys([p.strip() for p in p_matches if len(p.strip()) > 3]))[:3]
            if clean_projects: valid_insights.append(f"Demonstrated project portfolio: {', '.join(clean_projects)}")
        if certs:
            c_names = [str(c.get('name') or '?') if isinstance(c, dict) else str(c) for c in certs]
            valid_insights.append(f"Verified certifications: {', '.join(c_names)}")
        if unmapped:
            valid_insights.append(f"Extra capabilities beyond JD: {', '.join([str(u).title() for u in list(unmapped)[:4]])}")
        if not valid_insights: valid_insights.append("Candidate demonstrates active hands-on experience.")

    # FIX: `return {` was missing entirely -- this function fell straight
    # from the if-block above into dict key-value pairs with no opening
    # brace, which is a fatal syntax error (the notebook couldn't even
    # load, let alone run).
    #
    # FIX: candidate_name and exp_source were both computed as local
    # variables earlier in this function but never actually included in
    # the return dict -- and extraction_engine (also required by
    # print_report()) was never computed at all. All three added below.
    extraction_engine = f"Parallel NLP + LLM ({llm_data.get('llm_status', 'unavailable')})"

    return {
        "candidate_name": candidate_name,
        "extraction_engine": extraction_engine,
        "experience_source": exp_source,
        "resume_skills": resume_skills,
        "jd_skills": jd_skills,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "unmapped_skills": list(unmapped),
        "certifications": certs,
        "skill_match_score": skill_score,
        "matched_skill_count": len(matched_skills),
        "missing_skill_count": len(missing_skills),
        "resume_experience": resume_exp,
        "jd_experience": jd_exp,
        "experience_match_score": exp_score,
        "experience_ratio": exp_ratio,
        "is_overqualified": overqualified,
        "resume_experience_mentioned": resume_exp_mentioned,
        "jd_experience_mentioned": jd_exp_mentioned,
        "unstated_experience_for_senior_role": unstated_exp,
        "resume_education": resume_edu,
        "resume_education_level": resume_edu_level,
        "semantic_similarity_score": sim_score,
        "resume_domain": resume_domain,
        "jd_domain": jd_domain,
        "domain_match": domain_match_val,
        "value_addition_insights": valid_insights,  # FIX: was llm_data.get(...) directly, discarding the fallback logic above
        "dynamic_recommendations": recs
    }


In [8]:
# Cell 7: MODEL TRAINING & EVALUATION ENGINE (DATASET LOADER)
def build_dataset(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        r_raw = str(row.get("resume_text", row.get("Resume", row.get("resume", ""))))
        j_raw = str(row.get("job_description", row.get("JD", row.get("jd", ""))))
        label_raw = str(row.get("match_label", row.get("label", row.get("Label", row.get("Match", "unknown"))))).strip().lower()
        
        resume_clean = clean_text(r_raw)
        jd_clean = clean_text(j_raw)
        
        r_skills = extract_skills(resume_clean)
        j_skills = extract_skills(jd_clean)
        matched = get_matched_skills(r_skills, j_skills)
        missing = get_missing_skills(r_skills, j_skills)
        
        r_exp = extract_resume_experience(resume_clean)
        j_exp = extract_jd_experience(jd_clean)
        exp_score = calculate_experience_match_score(r_exp, j_exp)
        exp_ratio = calculate_experience_ratio(r_exp, j_exp)
        r_exp_m = int(not pd.isna(r_exp))
        j_exp_m = int(not pd.isna(j_exp))
        unstated = experience_unknown_for_senior_role(r_exp_m, j_exp)
        
        r_dom = identify_domain(r_skills)
        j_dom = identify_domain(j_skills)
        dom_match = int(r_dom == j_dom and r_dom != "Unknown")
        sim = semantic_similarity(resume_clean, jd_clean)
        
        records.append({
            "skill_match_score": skill_match_score(r_skills, j_skills),
            "matched_skill_count": len(matched),
            "missing_skill_count": len(missing),
            "experience_match_score": exp_score,
            "experience_ratio": exp_ratio,
            "resume_experience_mentioned": r_exp_m,
            "jd_experience_mentioned": j_exp_m,
            "unstated_experience_for_senior_role": unstated,
            "semantic_similarity_score": sim,
            "domain_match": dom_match,
            "label": label_raw
        })
    return pd.DataFrame(records)

csv_path = "resumeJD2_pairs.csv"
if not os.path.exists(csv_path) and os.path.exists("/content/resumeJD2_pairs.csv"):
    csv_path = "/content/resumeJD2_pairs.csv"

if not os.path.exists(csv_path):
    try:
        from google.colab import files
        print("resumeJD2_pairs.csv not found in current directory. Please upload:")
        uploaded_csv = files.upload()
        csv_path = list(uploaded_csv.keys())[0]
    except ImportError:
        pass

if not os.path.exists(csv_path):
    raise FileNotFoundError("Could not find 'resumeJD2_pairs.csv'. Please upload 'resumeJD2_pairs.csv' into Google Colab.")

raw_df = pd.read_csv(csv_path)
print(f"Building dataset features from historical pairs ({csv_path})...")
feature_df = build_dataset(raw_df)

X = feature_df[FEATURES].copy()
y = feature_df["label"]

X_full = X.copy()
X_full["experience_match_score"] = impute_experience_score(X_full["experience_match_score"])
X_full["experience_ratio"] = X_full["experience_ratio"].fillna(EXPERIENCE_UNKNOWN_SENTINEL)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y), 1):
    X_train, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    clf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_val)
    acc = accuracy_score(y_val, preds)
    cv_scores.append(acc)

print(f"\n5-Fold Stratified CV Mean Accuracy: {np.mean(cv_scores):.2%}")

final_model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE)
final_model.fit(X_full, y)

train_preds = final_model.predict(X_full)
print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE METRICS:")
print("="*60)
print(f"Accuracy : {accuracy_score(y, train_preds):.2%}")
print(f"F1 Score : {f1_score(y, train_preds, average='weighted'):.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y, train_preds))
print("="*60)

joblib.dump(final_model, "resume_screening_rf_v1.joblib")
print("Saved trained model artifact: 'resume_screening_rf_v1.joblib'")

explainer = shap.TreeExplainer(final_model)


Building dataset features from historical pairs (resumeJD2_pairs.csv)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


5-Fold Stratified CV Mean Accuracy: 60.80%

FINAL MODEL PERFORMANCE METRICS:
Accuracy : 72.80%
F1 Score : 0.7133

Detailed Classification Report:
               precision    recall  f1-score   support

        match       0.74      0.82      0.78       158
     no match       0.73      0.88      0.80       190
partial match       0.69      0.43      0.53       152

     accuracy                           0.73       500
    macro avg       0.72      0.71      0.70       500
 weighted avg       0.72      0.73      0.71       500

Saved trained model artifact: 'resume_screening_rf_v1.joblib'


In [9]:
# ============================================================
# evaluate_candidate_report -- also missing entirely, but called in
# Cell 9. Ties run_parallel_hybrid_pipeline's extracted features to the
# trained Random Forest (final_model) and SHAP explainer from Cell 7,
# producing every key print_report() expects (match_score_pct,
# status_decision, needs_human_review, matched_skills_pct).
# ============================================================

def evaluate_candidate_report(resume_text_raw: str, jd_text_raw: str) -> dict:
    base = run_parallel_hybrid_pipeline(resume_text_raw, jd_text_raw)

    X_row = pd.DataFrame([{f: base[f] for f in FEATURES}])
    X_row["experience_match_score"] = impute_experience_score(X_row["experience_match_score"])
    X_row["experience_ratio"] = X_row["experience_ratio"].fillna(EXPERIENCE_UNKNOWN_SENTINEL)

    classes = list(final_model.classes_)
    positive_candidates = ["match", "yes", "fit", "suitable", 1, "1"]
    match_idx = next((classes.index(c) for c in positive_candidates if c in classes), None)
    if match_idx is None:
        match_idx = int(np.argmax(final_model.predict_proba(X_row)[0]))

    proba = final_model.predict_proba(X_row)[0]
    confidence = float(proba[match_idx])
    needs_review = confidence < CONFIDENCE_THRESHOLD
    status_decision = "RELIABLE MATCH FOR JD" if not needs_review else "UNDER HUMAN REVIEW"

    m_count = len(base["matched_skills"])
    tot_req = m_count + len(base["missing_skills"])
    matched_pct = round((m_count / tot_req) * 100, 1) if tot_req > 0 else 0.0

    reasons = []
    try:
        shap_values = explainer.shap_values(X_row)
        if isinstance(shap_values, list):
            sv = shap_values[match_idx][0]
        else:
            sv = shap_values[0]
            if hasattr(sv, "ndim") and sv.ndim > 1:
                sv = sv[:, match_idx]
        contribs = sorted(zip(FEATURES, sv), key=lambda x: -abs(x[1]))[:3]
        for feat, val in contribs:
            label = FEATURE_LABELS.get(feat, feat)
            direction = "increased" if val > 0 else "decreased"
            reasons.append(f"{label} {direction} match confidence")
    except Exception:
        pass  # explainability is a bonus, not load-bearing -- never crash the report over it

    base.update({
        "confidence": confidence,
        "match_score_pct": round(confidence * 100, 1),
        "needs_human_review": needs_review,
        "status_decision": status_decision,
        "matched_skills_pct": matched_pct,
        "reasons": reasons,
    })
    return base


In [ ]:
def print_report(report: dict):
    m_count = len(report['matched_skills'])
    tot_req = m_count + len(report['missing_skills'])
    confidence_pct = report.get('match_score_pct', report.get('confidence', 0) * 100)
    is_review = report.get('needs_human_review', confidence_pct < CONFIDENCE_THRESHOLD * 100)
    status = report.get('status_decision', 'UNDER HUMAN REVIEW' if is_review else 'RELIABLE MATCH FOR JD')

    def fmt_years(val):
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return "Not stated / unverified"
        return f"{val:.1f} yrs"

    def fmt_pct(val):
        try:
            return f"{float(val):.1%}" if float(val) <= 1 else f"{float(val):.1f}%"
        except (TypeError, ValueError):
            return "N/A"

    W = 78
    print("\n" + "=" * W)
    print("  CANDIDATE EVALUATION REPORT".center(W))
    print("=" * W)
    print(f"  Candidate Name       : {report['candidate_name'].upper()}")
    print(f"  Model Confidence     : {confidence_pct:.1f}%  (Decision Threshold: {CONFIDENCE_THRESHOLD * 100:.0f}%)")
    print(f"  Final Decision       : {status}")
    print(f"  Human Review Needed  : {'YES - score below threshold, please verify manually' if is_review else 'NO - confidently matched to JD'}")
    print("-" * W)
    print(f"  Extraction Engine    : {report['extraction_engine']}")
    print(f"  Experience Source    : {report['experience_source']}")

    print("\n" + "-" * W)
    print("  SKILLS ANALYSIS")
    print("-" * W)
    print(f"  Overall Skill Match  : {m_count}/{tot_req} required skills matched ({report.get('matched_skills_pct', 0)}%)")
    if report.get('matched_skills'):
        print(f"  Matched Skills       : {', '.join(s.title() for s in report['matched_skills'])}")
    if report.get('missing_skills'):
        print(f"  Missing Skills       : {', '.join(s.title() for s in report['missing_skills'])}")
    if report.get('unmapped_skills'):
        print(f"  Bonus Skills         : {', '.join(str(s).title() for s in report['unmapped_skills'])}  (not required by JD, but a plus)")

    print("\n" + "-" * W)
    print("  EXPERIENCE & EDUCATION")
    print("-" * W)
    print(f"  Candidate Experience : {fmt_years(report.get('resume_experience'))}")
    print(f"  JD Requires          : {fmt_years(report.get('jd_experience'))}")
    if report.get('is_overqualified'):
        print(f"  Overqualification    : YES - candidate experience significantly exceeds JD requirement")
    print(f"  Education Level      : {report.get('resume_education_level', 'Unknown')}")
    if report.get('resume_education'):
        print(f"  Education Details    : {report['resume_education']}")

    print("\n" + "-" * W)
    print("  DOMAIN & SEMANTIC FIT")
    print("-" * W)
    print(f"  Candidate Domain     : {report.get('resume_domain', 'Unknown')}")
    print(f"  JD Domain            : {report.get('jd_domain', 'Unknown')}")
    print(f"  Domain Match         : {'YES' if report.get('domain_match') else 'NO'}")
    print(f"  Semantic Similarity  : {fmt_pct(report.get('semantic_similarity_score', 0))}")

    if report.get('certifications'):
        print("\n" + "-" * W)
        print("  CERTIFICATIONS")
        print("-" * W)
        for c in report['certifications']:
            name = c.get('name') if isinstance(c, dict) else str(c)
            print(f"  - {name}")

    if report.get('reasons'):
        print("\n" + "-" * W)
        print("  TOP FACTORS DRIVING THIS DECISION (model explainability)")
        print("-" * W)
        for r in report['reasons']:
            print(f"  - {r}")

    if report.get('value_addition_insights'):
        print("\n" + "-" * W)
        print("  VALUE-ADD INSIGHTS")
        print("-" * W)
        for v in report['value_addition_insights']:
            print(f"  - {v}")

    if report.get('dynamic_recommendations'):
        print("\n" + "-" * W)
        print("  SUGGESTED UPSKILLING (for missing JD skills)")
        print("-" * W)
        for rec in report['dynamic_recommendations']:
            print(f"  - {str(rec['skill']).title():<22} -> {rec['resource']}")

    print("\n" + "=" * W + "\n")

In [12]:
# Cell 9: CANDIDATE EVALUATION (LOCAL & COLAB)
import os

def get_file(prompt_name: str) -> str:
    """Works in either environment:
    - Google Colab: pops a real upload button.
    - Local Jupyter / VS Code: asks you to type/paste a file path
      (no native upload widget available without extra dependencies)."""
    try:
        from google.colab import files
        print(f"\n[Colab] Please upload {prompt_name}:")
        uploaded = files.upload()
        if uploaded:
            return list(uploaded.keys())[0]
    except ImportError:
        pass  # not running in Colab

    while True:
        path = input(f"Enter the file path for {prompt_name} (PDF/DOCX/image): ").strip().strip('"')
        if os.path.exists(path):
            return path
        print(f"  Could not find '{path}' -- check the path and try again.")

res_file = get_file("the candidate's RESUME")
jd_file = get_file("the JOB DESCRIPTION")

if os.path.exists(res_file) and os.path.exists(jd_file):
    print(f"[+] Processing Candidate Resume: {res_file}")
    print(f"[+] Processing Job Description: {jd_file}")
    res_text, err1 = safe_extract_text_from_file(res_file)
    jd_text, err2 = safe_extract_text_from_file(jd_file)
    if err1 or err2:
        print(f"Error reading files: {err1 or err2}")
    else:
        report = evaluate_candidate_report(res_text, jd_text)
        print_report(report)
else:
    print("[!] Could not locate one or both files.")


Enter the file path for the candidate's RESUME (PDF/DOCX/image):  "C:\Users\Admin\Downloads\Ayush Sharma Resume.pdf"
Enter the file path for the JOB DESCRIPTION (PDF/DOCX/image):  "C:\Users\Admin\Downloads\Celebal_JD.pdf"


[+] Processing Candidate Resume: C:\Users\Admin\Downloads\Ayush Sharma Resume.pdf
[+] Processing Job Description: C:\Users\Admin\Downloads\Celebal_JD.pdf

  CANDIDATE NAME     : AYUSH SHARMA
  MODEL CONFIDENCE   : 49.5%  (Threshold: 55%)
  STATUS DECISION    : UNDER HUMAN REVIEW
  HUMAN REVIEW NEEDED: YES (Score below 55% threshold)
  ENGINE             : Parallel NLP + LLM (Ollama llama3.2 (JSON mode))
  EXP SOURCE         : LLM-extracted dates, NLP-computed total (Tier 3)
  SKILL MATCH SUMMARY: 8/14 required skills matched (57.1%)

